In [3]:
import sys
print(sys.executable)


d:\projects\Langchain module tools and agents\langchain_env\Scripts\python.exe


In [1]:
from datasets import load_dataset
from dotenv import load_dotenv, find_dotenv
import pinecone
from pinecone import Pinecone, ServerlessSpec
import os
from sentence_transformers import SentenceTransformer

d:\projects\Langchain module tools and agents\langchain_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fw = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)

d:\projects\Langchain module tools and agents\langchain_env\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\muham\.cache\huggingface\hub\datasets--HuggingFaceFW--fineweb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
fw

IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count'],
    num_shards: 15
})

In [4]:
fw.features

{'text': Value('string'),
 'id': Value('string'),
 'dump': Value('string'),
 'url': Value('string'),
 'date': Value('string'),
 'file_path': Value('string'),
 'language': Value('string'),
 'language_score': Value('float64'),
 'token_count': Value('int64')}

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 898.54it/s]


In [6]:
load_dotenv(find_dotenv(), override=True)

True

In [7]:
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"), environment=os.environ.get("PINECONE_ENV"))

In [8]:
pc.list_indexes()

[
    {
        "name": "vectordb",
        "metric": "cosine",
        "host": "vectordb-fh1kny3.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 512,
        "deletion_protection": "disabled",
        "tags": {
            "embedding_model": "text-embedding-3-small"
        }
    }
]

In [9]:
pc.create_index(
    name="text",
    dimension=model.get_sentence_embedding_dimension(),
    metric="cosine",
    spec=ServerlessSpec(
        cloud='aws',
        region='us-east-1'
    )
)

C:\Users\muham\AppData\Local\Temp\ipykernel_37868\2249645534.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension=model.get_sentence_embedding_dimension(),


{
    "name": "text",
    "metric": "cosine",
    "host": "text-fh1kny3.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [11]:
index = pc.Index("text")
# Set the number of records to process
subset_size = 10000

vectors_to_upsert = []

for i, item in enumerate(fw):
    if i >= subset_size:
        break

    text = item["text"]
    unique_id = str(item["id"])
    language = item["language"]

    # Compute dense vector representations
    embedding = model.encode(text, show_progress_bar=False).tolist()

    # Define metadata payload
    metadata = {"language": language}

    # Append tuple matching Pinecone's expected structure: (id, vector, metadata)
    vectors_to_upsert.append((unique_id, embedding, metadata))

# Upload records in chunks
batch_size = 1000
for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i : i + batch_size]
    index.upsert(vectors=batch)

print("Subset of data upserted to Pinecone index.")

Subset of data upserted to Pinecone index.
